In [1]:
import geopandas as gpd

In [4]:
gdf = gpd.read_file(
    "../data/raw/SA4_2026_AUST_SHP_GDA2020/SA4_2026_AUST_GDA2020.shp"
)

print(gdf.head())
print(gdf.columns)

  SA4_CODE26               SA4_NAME26 CHG_FLAG26  CHG_LBL26 GCC_CODE26  \
0        101           Capital Region          0  No change      1RNSW   
1        102            Central Coast          0  No change      1GSYD   
2        103             Central West          0  No change      1RNSW   
3        104  Coffs Harbour - Grafton          0  No change      1RNSW   
4        105       Far West and Orana          0  No change      1RNSW   

       GCC_NAME26 STE_CODE26       STE_NAME26 AUS_CODE26 AUS_NAME26  \
0     Rest of NSW          1  New South Wales        AUS  Australia   
1  Greater Sydney          1  New South Wales        AUS  Australia   
2     Rest of NSW          1  New South Wales        AUS  Australia   
3     Rest of NSW          1  New South Wales        AUS  Australia   
4     Rest of NSW          1  New South Wales        AUS  Australia   

    AREASQKM26                                           geometry  
0   51896.2445  MULTIPOLYGON (((150.08176 -36.377, 150.08159

In [5]:
print(gdf.crs)

EPSG:7844


In [6]:
import pandas as pd

ev = pd.read_csv("../data/raw/ev_20251216.csv")

print(ev.head())
print(ev.columns.tolist())
print(ev.shape)

   OBJECTID Station_name                               Station_address  \
0       NaN          NaN                          , Muswellbrook, 2333   
1       NaN          NaN               01 Wallgrove Road, Sydney, 2766   
2       NaN          NaN  1 - 7 Ross St, Wilcannia NSW 2836, Australia   
3       NaN          NaN                    1 Balfour St, Sydney, 2070   
4       NaN          NaN                     1 Bay Ln, Byron Bay, 2481   

    Operator  Number_of_plugs Charger_Type Charger_rating   Latitude  \
0       EVUp                2           AC          22 kW -32.262242   
1         BP                4           DC         150 kW -33.811004   
2       NRMA                4           DC          50 kW -30.511874   
3  Chargefox                7           AC          22 kW -33.774101   
4      Tesla                2           AC          19 kW -28.641819   

    Longitude                        LGANAME PCODE  \
0  150.890139     Muswellbrook Shire Council  2333   
1  150.849597 

In [7]:
ev_gdf = gpd.GeoDataFrame(
    ev,
    geometry=gpd.points_from_xy(ev["Longitude"], ev["Latitude"]),
    crs="EPSG:4326"
)

print(ev_gdf.head())
print(ev_gdf.crs)

   OBJECTID Station_name                               Station_address  \
0       NaN          NaN                          , Muswellbrook, 2333   
1       NaN          NaN               01 Wallgrove Road, Sydney, 2766   
2       NaN          NaN  1 - 7 Ross St, Wilcannia NSW 2836, Australia   
3       NaN          NaN                    1 Balfour St, Sydney, 2070   
4       NaN          NaN                     1 Bay Ln, Byron Bay, 2481   

    Operator  Number_of_plugs Charger_Type Charger_rating   Latitude  \
0       EVUp                2           AC          22 kW -32.262242   
1         BP                4           DC         150 kW -33.811004   
2       NRMA                4           DC          50 kW -30.511874   
3  Chargefox                7           AC          22 kW -33.774101   
4      Tesla                2           AC          19 kW -28.641819   

    Longitude                        LGANAME PCODE  \
0  150.890139     Muswellbrook Shire Council  2333   
1  150.849597 

In [8]:
ev_gdf = ev_gdf.to_crs(gdf.crs)

print(ev_gdf.crs)
print(gdf.crs)

EPSG:7844
EPSG:7844


In [9]:
joined = gpd.sjoin(
    ev_gdf,
    gdf[["SA4_CODE26", "SA4_NAME26", "STE_NAME26", "geometry"]],
    how="left",
    predicate="within"
)

print(joined[
    ["Station_name", "Latitude", "Longitude",
     "SA4_CODE26", "SA4_NAME26", "STE_NAME26"]
].head(10))

print("Original EV rows:", len(ev_gdf))
print("Joined rows:", len(joined))
print("No SA4 match:", joined["SA4_CODE26"].isna().sum())

  Station_name   Latitude   Longitude SA4_CODE26  \
0          NaN -32.262242  150.890139        106   
1          NaN -33.811004  150.849597        116   
2          NaN -30.511874  151.669395        110   
3          NaN -33.774101  151.167035        121   
4          NaN -28.641819  153.613633        112   
5          NaN -33.883504  151.194433        117   
6          NaN -28.276903  153.577078        112   
7          NaN -33.841037  151.242245        121   
8          NaN -33.841418  151.242631        121   
9          NaN -33.871368  151.213741        117   

                          SA4_NAME26       STE_NAME26  
0        Hunter Valley exc Newcastle  New South Wales  
1                 Sydney - Blacktown  New South Wales  
2         New England and North West  New South Wales  
3  Sydney - North Sydney and Hornsby  New South Wales  
4                   Richmond - Tweed  New South Wales  
5      Sydney - City and Inner South  New South Wales  
6                   Richmond - Twee

In [10]:
unmatched = joined[joined["SA4_CODE26"].isna()]

print(unmatched[
    ["Station_name", "Station_address",
     "Latitude", "Longitude", "LGANAME", "PCODE"]
])

     Station_name                               Station_address  Latitude  \
1833          NaN  1 Sandy Bay Rd\nClontarf NSW 2093\nAustralia -33.80467   

      Longitude                   LGANAME PCODE  
1833  151.25284  Northern Beaches Council  2093  


In [11]:
# Get the unmatched EV charger
point = unmatched.geometry.iloc[0]

# Convert both datasets to a projected CRS for distance calculation
point_proj = gpd.GeoSeries([point], crs="EPSG:7844").to_crs("EPSG:7856")
sa4_proj = gdf.to_crs("EPSG:7856")

# Calculate distance to every SA4 polygon
distances = sa4_proj.geometry.distance(point_proj.iloc[0])

# Find the nearest SA4
nearest_idx = distances.idxmin()

print("Nearest SA4:", sa4_proj.loc[nearest_idx, "SA4_NAME26"])
print("State:", sa4_proj.loc[nearest_idx, "STE_NAME26"])
print("Distance:", round(distances.loc[nearest_idx], 2), "metres")

Nearest SA4: Sydney - Northern Beaches
State: New South Wales
Distance: 1.77 metres


In [12]:
# Fill the unmatched charger with its nearest SA4
joined.loc[unmatched.index, "SA4_CODE26"] = gdf.loc[nearest_idx, "SA4_CODE26"]
joined.loc[unmatched.index, "SA4_NAME26"] = gdf.loc[nearest_idx, "SA4_NAME26"]
joined.loc[unmatched.index, "STE_NAME26"] = gdf.loc[nearest_idx, "STE_NAME26"]

# Check the result
print(joined.loc[unmatched.index, [
    "Station_address",
    "SA4_CODE26",
    "SA4_NAME26",
    "STE_NAME26"
]])

print("No SA4 match:", joined["SA4_CODE26"].isna().sum())

                                   Station_address SA4_CODE26  \
1833  1 Sandy Bay Rd\nClontarf NSW 2093\nAustralia        122   

                     SA4_NAME26       STE_NAME26  
1833  Sydney - Northern Beaches  New South Wales  
No SA4 match: 0
